# Floating-Point Arithmetic

| Difficulty | ████░░░░░░ 4/10 |
| :--- | :--- |
| Prerequisites | 01.01–01.10, 02.01–02.10 |
| Time | ~55 min |
| Colab GPU | Not needed |

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/NiravRVaghasiya/maths-for-ai/blob/main/06_Numerical_Methods/01_floating_point_arithmetic.ipynb)

---

## 🎯 Learning Objective

Understand the IEEE 754 floating-point standard and the tradeoffs between float32, float16, and bfloat16.

---

## 📐 Theory

Every number in this repo's code cells has, until now, been treated as "just a number." This
notebook opens the hood: computers don't store real numbers exactly, they store a finite
approximation, and the rules of that approximation (IEEE 754) shape everything from a single
`0.1 + 0.2` to whether a billion-parameter model trains without crashing.

### The IEEE 754 layout

A floating-point number is stored as three fields packed into a fixed number of bits:

$$\text{value} = (-1)^{\text{sign}} \times (1.\text{mantissa})_2 \times 2^{\text{exponent} - \text{bias}}$$

- **Sign** (1 bit): $0$ for positive, $1$ for negative.
- **Exponent** (8 bits for float32): stored with a **bias** ($127$ for float32) so it can
  represent both very large and very small magnitudes using only unsigned bits.
  Unbiased exponent = stored exponent $-$ bias.
- **Mantissa** (23 bits for float32): the fractional part after an implicit leading $1$
  (normalized form), giving $1 + \text{mantissa}/2^{23}$ as the significant digits.

float32 (`np.float32`) uses 1+8+23 = 32 bits total. float16 (`np.float16`) uses 1+5+10 = 16
bits — far fewer exponent bits (smaller range) *and* fewer mantissa bits (less precision).
**bfloat16** is Google's deliberate compromise: 1+8+7 = 16 bits, keeping float32's *full
exponent range* (same 8 exponent bits) while sacrificing mantissa precision instead. This
design choice is not arbitrary — it directly determines what breaks when you switch precisions,
covered below.

### Machine epsilon: the smallest meaningful gap

**Machine epsilon** $\epsilon$ is the smallest number such that $1 + \epsilon \ne 1$ in that
format's arithmetic — it quantifies the format's *relative* precision. float32's $\epsilon
\approx 1.19\times 10^{-7}$; float16's $\epsilon \approx 9.77\times 10^{-4}$, about $8000\times$
coarser. Critically, precision is *relative to magnitude*, not absolute: the gap between
representable numbers near $1000.0$ is much larger than the gap near $0.001$, because the
exponent field, not just the mantissa, determines spacing.

### Overflow and underflow

**Overflow**: a value's true magnitude exceeds what the exponent field can represent, so it
rounds to $\pm\infty$. float16's largest representable value is $65504$ — a value like $70000$
overflows immediately, no exotic computation required. **Underflow**: a value is too *small* in
magnitude to represent even with the smallest exponent, so it rounds to exactly $0$. float16's
smallest positive normal value is about $6.1\times 10^{-5}$; anything much smaller silently
becomes zero. Both failure modes are format-specific: the exact same Python float (float64
internally) can overflow when cast to float16 but be completely unremarkable in float32.

### Why this matters before anything else in Module 06

Every subsequent notebook in this module — stable softmax, gradient checking, mixed-precision
training, transformer numerical issues — is really asking one of two questions about a
computation: "does an intermediate value overflow/underflow in this format?" or "does this
operation lose more precision than the format's $\epsilon$ can absorb?" Understanding the bit
layout here is what makes those later questions answerable by calculation instead of guesswork.

---

In [ ]:
# Setup
import numpy as np
import struct
import matplotlib.pyplot as plt
%matplotlib inline
plt.style.use('seaborn-v0_8-whitegrid')

## 👁️ Visual Intuition

Plotting the *gap* between consecutive representable floats as a function of magnitude makes
"precision is relative, not absolute" visible: the gap grows in discrete jumps every time the
exponent increments, and float16's gaps are dramatically coarser than float32's at every scale.

In [ ]:
# The gap to the NEXT representable float, as a function of magnitude, for float32 vs float16.
# np.spacing(x) returns exactly this: the distance from x to the next representable float.
magnitudes = np.logspace(-2, 4, 300)
gaps_32 = np.array([np.spacing(np.float32(m)) for m in magnitudes])
gaps_16 = np.array([np.spacing(np.float16(m)) for m in magnitudes])

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
axes[0].loglog(magnitudes, gaps_32, color="#4C72B0", label="float32")
axes[0].loglog(magnitudes, gaps_16, color="#C44E52", label="float16")
axes[0].set_xlabel("magnitude of the number"); axes[0].set_ylabel("gap to next representable float")
axes[0].set_title("Precision is RELATIVE to magnitude:\nboth gaps grow with the number itself")
axes[0].legend()

# Ratio: how much coarser is float16 than float32 at the SAME magnitude? Should be
# roughly constant (matching the ~8000x machine-epsilon ratio from the Theory section)
ratio = gaps_16 / gaps_32
axes[1].semilogx(magnitudes, ratio, color="#55A868")
axes[1].axhline(np.finfo(np.float16).eps / np.finfo(np.float32).eps, color="k", ls="--",
                 label="eps16 / eps32 (theoretical)")
axes[1].set_xlabel("magnitude"); axes[1].set_ylabel("float16 gap / float32 gap")
axes[1].set_title("float16 is consistently ~8000x coarser\nthan float32, at every magnitude")
axes[1].legend()
plt.tight_layout()
plt.show()

print(f"machine epsilon: float32={np.finfo(np.float32).eps:.3e}, float16={np.finfo(np.float16).eps:.3e}")
print(f"ratio: {np.finfo(np.float16).eps / np.finfo(np.float32).eps:.1f}x coarser")

## 🐍 Implementation from Scratch

We use Python's `struct` module to unpack a float32's raw bits, extract sign/exponent/mantissa
by hand, and reconstruct the original value from those three fields — confirming the IEEE 754
formula from the Theory section bit-for-bit, then demonstrate real overflow and underflow.

In [ ]:
def decompose_float32(value):
    """Unpack a Python float's IEEE 754 float32 bit pattern into (sign, exponent, mantissa),
    using struct to get the raw 32-bit integer representation, then bit-shifting/masking."""
    packed = struct.pack('>f', value)          # big-endian 4 bytes, the raw float32 bits
    bits = struct.unpack('>I', packed)[0]      # same 4 bytes, read back as an unsigned int
    sign = (bits >> 31) & 1
    exponent = (bits >> 23) & 0xFF             # 8 bits, still BIASED (add -127 to unbias)
    mantissa = bits & 0x7FFFFF                 # low 23 bits
    return sign, exponent, mantissa, bits

def reconstruct_from_bits(sign, exponent, mantissa):
    # Normalized form: value = (-1)^sign * (1 + mantissa/2^23) * 2^(exponent-127)
    if exponent == 0:  # subnormal: no implicit leading 1, and a fixed minimum exponent
        return ((-1) ** sign) * (mantissa / 2**23) * 2**(-126)
    return ((-1) ** sign) * (1 + mantissa / 2**23) * 2 ** (exponent - 127)

test_values = [1.0, -1.0, 0.5, 100.0, 0.1]
print(f"{'value':>8} {'sign':>4} {'exp(biased)':>11} {'exp(true)':>9} {'mantissa (binary)':>25}")
for v in test_values:
    sign, exp, mant, bits = decompose_float32(v)
    print(f"{v:8} {sign:4d} {exp:11d} {exp-127:9d} {mant:025b}")
    reconstructed = reconstruct_from_bits(sign, exp, mant)
    assert np.isclose(reconstructed, v, rtol=1e-6), f"mismatch: {reconstructed} != {v}"
print("\nEvery reconstruction matches the original value -- the IEEE 754 formula is exact.")

# --- Overflow: float16's largest finite value is 65504; anything bigger becomes inf ---
print(f"\nfloat16 max finite value: {np.finfo(np.float16).max}")
big_value = np.float32(70000.0)
overflowed = big_value.astype(np.float16)
print(f"70000.0 cast from float32 to float16: {overflowed}  (overflowed to infinity)")

# --- Underflow: float16's smallest positive NORMAL value is ~6.1e-5 ---
print(f"\nfloat16 smallest positive normal value: {np.finfo(np.float16).tiny:.3e}")
small_value = np.float32(1e-8)
underflowed = small_value.astype(np.float16)
print(f"1e-8 cast from float32 to float16: {underflowed}  (underflowed to exactly zero)")
print(f"That same 1e-8 survives perfectly fine in float32: {small_value}")

## 🤖 Why This Matters for AI

Modern LLMs train almost exclusively in **bfloat16**, not float16, and the reason is exactly
the exponent-vs-mantissa tradeoff from the Theory section. A transformer's residual stream
(the running sum of every layer's output, recall `02.10`) can grow to activation magnitudes
in the thousands or more as depth increases; float16's narrow exponent range (max $65504$)
overflows to `inf` on values that bfloat16, sharing float32's full 8-bit exponent, handles
without blinking. The cost is precision: bfloat16 has fewer mantissa bits than float16, so it
distinguishes nearby values less finely. Below, we reproduce both effects directly: an
activation value that overflows in float16 but survives in bfloat16, and a pair of nearby
values that float16 can tell apart but bfloat16 cannot.

In [ ]:
import torch

# Simulate a residual stream whose magnitude grows across layers (a realistic pattern in
# unnormalized/under-normalized transformers) -- float16 overflows, bfloat16 does not
activations_fp32 = torch.tensor([100.0, 1000.0, 10000.0, 50000.0, 100000.0])
act_fp16 = activations_fp32.to(torch.float16)
act_bf16 = activations_fp32.to(torch.bfloat16)

print("Simulated residual-stream magnitude across layers (float32, ground truth):")
print(" ", activations_fp32.tolist())
print("Cast to float16:  ", act_fp16.tolist(), " <- overflowed to inf at the largest value")
print("Cast to bfloat16: ", act_bf16.tolist(), " <- stays finite everywhere, but less precise")

# Now the flip side: a difference float16 CAN represent but bfloat16 CANNOT, because
# bfloat16 sacrifices mantissa bits (7) for exponent range, vs float16's 10 mantissa bits
a = torch.tensor([1.0, 1.002])
a_fp16 = a.to(torch.float16)
a_bf16 = a.to(torch.bfloat16)
print(f"\nTwo close values in float32: {a.tolist()}")
print(f"Cast to float16:  {a_fp16.tolist()}  (still distinguishable: {a_fp16[0].item() != a_fp16[1].item()})")
print(f"Cast to bfloat16: {a_bf16.tolist()}  (distinguishable: {a_bf16[0].item() != a_bf16[1].item()})")
print("\nThis is the exact tradeoff transformer training makes: bfloat16 accepts coarser")
print("precision in exchange for float32's full dynamic range, because overflow (a hard crash)")
print("is a worse failure mode during training than a small, gradual loss of precision.")

## 🏋️ Exercises

### Warm-up
1. By hand, decompose $-8.5$ into its IEEE 754 float32 sign, biased exponent, and mantissa
   bits. Verify with `decompose_float32`.

### Practice
2. Find the smallest positive integer that, when cast to `np.float32`, no longer increments by
   $1$ when you add $1$ to it (i.e. `np.float32(n) + np.float32(1) == np.float32(n)`). Explain
   why this happens in terms of machine epsilon and the mantissa's fixed number of bits.

### Challenge
3. Simulate the residual-stream growth experiment in the AI section, but starting from a
   *deeper* stack (extend `activations_fp32` to 10 monotonically increasing values reaching
   $10^7$). Report at which layer float16 first overflows, and confirm bfloat16 survives the
   entire stack. Then explain why simply clipping activations (as in `04.08`'s gradient
   clipping) is a *different* fix than switching number formats, and why a real training setup
   might want both.

---

## 📚 Further Reading
- IEEE 754-2019 Standard for Floating-Point Arithmetic (summary: [Wikipedia: IEEE 754](https://en.wikipedia.org/wiki/IEEE_754))
- Kalamkar et al., ["A Study of BFLOAT16 for Deep Learning Training"](https://arxiv.org/abs/1905.12322)

---

*Next notebook: [Numerical Stability](02_numerical_stability.ipynb)*